##### Boiler-Plate TensorFlow Model.

> this is a boiler plate code from a google tutorial:

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> run on google colab</a>
  </td>
</table>

built on [keras](https://www.tensorflow.org/guide/keras/overview):

1. load a prebuilt dataset.
1. `model`:neural network machine learning model that classifies images.
2. train this neural network.
3. evaluate the accuracy of it.

this is a [google colab](https://colab.research.google.com/notebooks/welcome.ipynb) notebook. use the browser as your frontend for your notebook.  
> connect to a python runtime:
 > run the code in the notebook, per cell(shift + enter) or all of them at once.


## set up `tensorflow`

import tensorflow into your program this way:

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0-dev0+selfbuilt


LOCAL SETUP: 
> sadface :}
> see the [install guide](https://www.tensorflow.org/install) for setting up `tensorflow` locally.

> note: make sure you have upgraded to the latest `pip` to install the tensorflow package
> or use your system's way for your setup eg on arch you can install it w/ the commandline below
>
>
```sh
sudo pacman -S python-tensorflow
```

## load MNIST dataset

load and prepare the MNIST dataset. the pixel values of the images range from 0 through 255. you can scale these values to a range of 0 to 1 by dividing the values by `255.0`. this also converts the sample data from integers to floating-point numbers:

In [2]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

## machine learning model

you can decide to go with a `tf.keras.Sequential` model:

In [3]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Input(shape=(28, 28)),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10)
])

[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) is useful for stacking layers where each layer has one input [tensor](https://www.tensorflow.org/guide/tensor) and one output tensor. layers are functions with a known mathematical structure that can be reused and have trainable variables. most models are composed of layers. this model uses the [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense), and [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) layers.

the model returns a vector of [logits](https://developers.google.com/machine-learning/glossary#logits) or [log-odds](https://developers.google.com/machine-learning/glossary#log-odds) scores, one for each class.

In [4]:
predictions = model(x_train[:1]).numpy()
predictions

array([[ 0.148776  ,  0.01754591, -0.12847546,  0.12291473, -0.54330873,
         0.19376889, -0.22354622, -0.20450756,  0.25386202,  0.3415743 ]],
      dtype=float32)

the `tf.nn.softmax` function converts these logits to *probabilities* for each class: 

In [5]:
tf.nn.softmax(predictions).numpy()

array([[0.11272869, 0.09886486, 0.085433  , 0.10985076, 0.05642426,
        0.1179165 , 0.07768496, 0.07917815, 0.12521972, 0.13669911]],
      dtype=float32)

NOTE: It is possible to bake the `tf.nn.softmax` function into the activation function for the last layer of the network. while this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output. 

define a loss function for training using `losses.SparseCategoricalCrossentropy`:

In [6]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

the loss function takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. this loss is equal to the negative log probability of the true class: The loss is zero if the model is sure of the correct class.

this untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to `-tf.math.log(1/10) ~= 2.3`.

In [7]:
loss_fn(y_train[:1], predictions).numpy()

np.float32(2.1377783)

before you start training, configure and compile the model using Keras `Model.compile`. Set the [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) class to `adam`, set the `loss` to the `loss_fn` function you defined earlier, and specify a metric to be evaluated for the model by setting the `metrics` parameter to `accuracy`.

In [8]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## train + evaluate model

use the `Model.fit` method to adjust your model parameters and minimize the loss: 

In [9]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5


2025-10-21 05:11:26.405685: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 188160000 exceeds 10% of free system memory.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9126 - loss: 0.3014
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9570 - loss: 0.1439
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9682 - loss: 0.1071
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9723 - loss: 0.0889
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9759 - loss: 0.0771


the `Model.evaluate` method checks the model's performance, usually on a [validation set](https://developers.google.com/machine-learning/glossary#validation-set) or [test set](https://developers.google.com/machine-learning/glossary#test-set).

In [10]:
model.evaluate(x_test,  y_test, verbose=2)

313/313 - 1s - 2ms/step - accuracy: 0.9761 - loss: 0.0771


[0.07710790634155273, 0.9761000275611877]

the image classifier is now trained to ~98% accuracy on this dataset

the model can return a probability, if can wrap the trained model, and attach the softmax to it:

In [11]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [12]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[1.9711686e-06, 3.4479601e-08, 2.0715612e-04, 1.9867669e-03,
        1.8819237e-09, 9.8903081e-07, 3.5798031e-12, 9.9772674e-01,
        1.8606890e-05, 5.7791862e-05],
       [2.9026881e-09, 4.6913092e-06, 9.9994504e-01, 4.9593007e-05,
        2.5759573e-14, 3.6685453e-08, 2.8008809e-08, 1.5077359e-14,
        5.8894386e-07, 1.3873345e-13],
       [4.5851075e-06, 9.9509579e-01, 1.0096544e-04, 1.4407208e-04,
        4.4446383e-04, 7.2322153e-05, 9.8383367e-05, 1.5742051e-03,
        2.4496366e-03, 1.5519237e-05],
       [9.9975187e-01, 3.4825458e-09, 1.3070434e-04, 1.3885488e-06,
        8.3446622e-07, 9.5323267e-06, 5.4636468e-05, 4.0430987e-06,
        2.9788138e-05, 1.7142618e-05],
       [1.8109016e-05, 7.6133624e-09, 4.0344708e-05, 6.3152065e-08,
        9.9795002e-01, 7.2531215e-08, 3.8346956e-05, 5.6516608e-05,
        6.4911393e-07, 1.8960352e-03]], dtype=float32)>

## end